[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# [模块 1](https://dataflowr.github.io/website/modules/1-intro-general-overview/)：用 CNN 做猫狗分类

为了演示[模块 1](https://dataflowr.github.io/website/modules/1-intro-general-overview/)中讲的深度学习流水线，我们将使用一个预训练模型参加 Kaggle 的 [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition) 竞赛。

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=1175)


训练集里有 25,000 张带标签的猫狗照片，测试集有 12,500 张，需要我们为这个竞赛打上标签。据 Kaggle 网站介绍，这个竞赛在 2013 年底发起时：*"**当时的最高水准**：现有文献表明，机器分类器在这项任务上可以取得 80% 以上的准确率"*。所以如果你能超过 80%，你就达到了 2013 年的前沿水平！


##  导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torchvision
from torchvision import models,transforms,datasets
import time
%matplotlib inline

这里可以看到，默认安装的是最新版 PyTorch。


In [ ]:
torch.__version__

In [ ]:
import sys
sys.version

检查 GPU 是否可用，如果不可用就换一个 [runtime](https://jovianlin.io/pytorch-with-gpu-in-google-colab/)。


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print('Using gpu: %s ' % torch.cuda.is_available())

## 下载数据


你可以直接从 Kaggle 下载完整数据集。

另外，Jeremy Howard（fast.ai）提供了一个 catvsdogs [数据集](http://files.fast.ai/data/examples/)的直接链接。他把猫和狗分到了不同的文件夹，并且也建了一个验证集文件夹。

为了测试（或者如果你在 CPU 上运行），你应该使用（较小的）sample 目录。


In [ ]:
%mkdir data
# 如果你在自己的电脑上运行这个 notebook，需要修改下面这一行
# 切换到存放数据集的 data 目录
%cd /content/data/
!wget http://files.fast.ai/data/examples/dogscats.tgz

In [ ]:
!tar -zxvf dogscats.tgz

In [ ]:
%ls

In [ ]:
%cd dogscats/
%ls

`dogscats` 文件夹内部子文件夹的结构对后面很重要：
```bash
.
├── test1 # 包含 12500 张猫和狗的图片
├── train
|   └── cats # 包含 11500 张猫的图片
|   └── dogs # 包含 11500 张狗的图片
├── valid
|   └── cats # 包含 1000 张猫的图片
|   └── dogs # 包含 1000 张狗的图片
├── sample
|   └── train
|       └── cats # 包含 8 张猫的图片
|       └── dogs # 包含 8 张狗的图片    
|   └── valid 
|       └── cats # 包含 4 张猫的图片
|       └── dogs # 包含 4 张狗的图片    
├── models # 空文件夹
```

可以看到，测试集的 12,500 张图片在 `test1` 子文件夹里；25,000 张带标签的图片被分成了训练集和验证集。

子文件夹 `sample` 只是为了在很小的数据集上确认代码能正常运行。


## 数据处理


In [ ]:
%cd ..

下面给出数据存放的路径。如果你在自己的电脑上运行这段代码，需要修改这个单元格。

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=1550)


In [ ]:
data_dir = '/content/data/dogscats'

`datasets` 是 `torchvision` 包中的一个类（见 [torchvision.datasets](http://pytorch.org/docs/master/torchvision/datasets.html)），负责数据加载。它内置了一个多线程加载器：从磁盘读取图片、按小批次（mini-batch）分组，并在网络的每次 _前向_/_反向_ 传播之后持续地把数据喂给 GPU。

图片在送入网络之前需要做一些预处理：尺寸必须统一为 $224\times 224 \times 3$，另外还要经过下面 normalize 变换所做的一些额外格式化（后面会解释）。


In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

imagenet_format = transforms.Compose([
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                normalize,
            ])

In [ ]:
dsets = {x: datasets.ImageFolder(os.path.join(data_dir, x), imagenet_format)
         for x in ['train', 'valid']}

In [ ]:
os.path.join(data_dir,'train')

在 jupyter notebook 中，用 `?` 可以调出交互式帮助


In [ ]:
?datasets.ImageFolder

可以看到 `datasets.ImageFolder` 有这些属性：classes、class_to_idx、imgs。

看看它们分别是什么？


In [ ]:
dsets['train'].classes

类别名称直接从文件夹结构推断出来：
```bash
├── train
|   └── cats
|   └── dogs
```


In [ ]:
dsets['train'].class_to_idx

标签 0 对应猫，1 对应狗。

下面可以看到，前 5 张 imgs 是（图片路径，标签）这样的二元组：


In [ ]:
dsets['train'].imgs[:5]

In [ ]:
dset_sizes = {x: len(dsets[x]) for x in ['train', 'valid']}
dset_sizes

和预期一致，训练集有 23,000 张图片，验证集有 2,000 张。

下面我们把类别存到变量 `dset_classes` 里：


In [ ]:
dset_classes = dsets['train'].classes

`torchvision` 包支持对输入数据做复杂的预处理/变换（比如归一化、裁剪、翻转、抖动）。借助 `torchvision.transforms.Compose` 函数可以把一系列变换组合成一个流水线，见 [torchvision.transforms](http://pytorch.org/docs/master/torchvision/transforms.html)


神奇的 `?` 帮助还可以用来找回你定义过但忘了的函数！


In [ ]:
?imagenet_format

这个归一化是从哪来的？

正如 [PyTorch 文档](https://pytorch.org/docs/stable/torchvision/models.html) 中所说，你要使用预训练模型。所有预训练模型都期望输入图片按同样的方式归一化，即形状为 (3 x H x W) 的 3 通道 RGB 图片小批次，其中 H 和 W 至少为 224。图片需要先加载到 [0, 1] 范围内，再用 `mean = [0.485, 0.456, 0.406]` 和 `std = [0.229, 0.224, 0.225]` 做归一化。


In [ ]:
loader_train = torch.utils.data.DataLoader(dsets['train'], batch_size=64, shuffle=True, num_workers=6)

In [ ]:
?torch.utils.data.DataLoader

In [ ]:
loader_valid = torch.utils.data.DataLoader(dsets['valid'], batch_size=5, shuffle=False, num_workers=6)

试着理解下面这个单元格在做什么？


In [ ]:
count = len(loader_valid)
inputs_try, labels_try = next(iter(loader_valid))

In [ ]:
labels_try

In [ ]:
inputs_try.shape

明白了：验证集包含 2,000 张图片，因此是 400 个大小为 5 的 batch。`labels_try` 保存第一个 batch 的标签，`inputs_try` 保存第一个 batch 的图片。

对计算机来说，一张图片是什么？


In [ ]:
inputs_try[0]

一张 3 通道 RGB 图片的形状是 (3 x H x W)。注意，由于归一化，元素可以是负的。


一个显示图片的小函数：


In [ ]:
def imshow(inp, title=None):
#   用于显示 Tensor 的 imshow。
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = np.clip(std * inp + mean, 0,1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)

In [ ]:
# 从验证数据的一个 batch 生成网格图像
out = torchvision.utils.make_grid(inputs_try)

imshow(out, title=[dset_classes[x] for x in labels_try])

In [ ]:
# 取一个训练数据 batch
inputs, classes = next(iter(loader_train))

n_images = 8

# 从 batch 生成网格图像
out = torchvision.utils.make_grid(inputs[0:n_images])

imshow(out, title=[dset_classes[x] for x in classes[0:n_images]])

## 创建 VGG 模型


torchvision 模块自带一个流行的 CNN 架构动物园，这些模型已经在 [ImageNet](http://www.image-net.org/)（120 万张训练图片）上训练过。第一次调用时，如果 `pretrained=True`，模型会从网上下载并保存到 `~/.torch/models`。
之后再调用会直接从本地读取。

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=2451)


In [ ]:
model_vgg = models.vgg16(weights='DEFAULT')

我们先用未经任何修改的 VGG 模型。为了解释结果，需要导入 1000 个 ImageNet 类别，下载地址：[https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json](https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json)


In [ ]:
!wget https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json

In [ ]:
import json

fpath = '/content/data/imagenet_class_index.json'

with open(fpath) as f:
    class_dict = json.load(f)
dic_imagenet = [class_dict[str(i)][1] for i in range(len(class_dict))]

In [ ]:
dic_imagenet[:4]

In [ ]:
inputs_try , labels_try = inputs_try.to(device), labels_try.to(device)

model_vgg = model_vgg.to(device)

In [ ]:
outputs_try = model_vgg(inputs_try)

In [ ]:
outputs_try

In [ ]:
outputs_try.shape

为了把网络的输出转成'概率'，我们让它经过一个 [Softmax 函数](https://en.wikipedia.org/wiki/Softmax_function)


In [ ]:
m_softm = nn.Softmax(dim=1)
probs = m_softm(outputs_try)
vals_try,preds_try = torch.max(probs,dim=1)

我们验证一下，确实得到了一个概率！


In [ ]:
torch.sum(probs,1)

In [ ]:
vals_try

In [ ]:
print([dic_imagenet[i] for i in preds_try.data])

In [ ]:
out = torchvision.utils.make_grid(inputs_try.data.cpu())

imshow(out, title=[dset_classes[x] for x in labels_try.data.cpu()])

### 修改最后一层，并让所有层都不计算梯度

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=2755)


In [ ]:
print(model_vgg)

课程后面会学到这些不同的 block 各有什么作用。现在只需知道：

- 卷积层用于寻找图片中中小尺寸的模式——对图片做局部分析
- 全连接（Dense）层用于把整张图片上的模式组合起来——对图片做全局分析
- 池化层做下采样——减小图片尺寸，同时提高学到的特征的（平移）不变性


![vgg16](https://dataflowr.github.io/notebooks/Module1/img/vgg16.png)

在这个实操例子中，我们的目标是使用已经训练好的模型，只改输出类别数。为此，把最后那个为 1000 类训练的 `nn.Linear` 层换成 2 类的。为了在训练时冻结其他层的权重，我们把 `requires_grad` 设为 `False`。这样反向传播时不会为这些层计算梯度，权重也就不会更新。只有这 2 类的输出层权重会被更新。


In [ ]:
for param in model_vgg.parameters():
    param.requires_grad = False
model_vgg.classifier._modules['6'] = nn.Linear(4096, 2)
model_vgg.classifier._modules['7'] = torch.nn.LogSoftmax(dim = 1)

PyTorch 关于 [LogSoftmax](https://pytorch.org/docs/stable/nn.html#logsoftmax) 的文档


In [ ]:
print(model_vgg.classifier)

把模型加载到 GPU 上。


In [ ]:
model_vgg = model_vgg.to(device)

## 训练全连接模块

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=2990)


### 创建损失函数和优化器

PyTorch 关于 [NLLLoss](https://pytorch.org/docs/stable/nn.html#nllloss) 和 [torch.optim 模块](https://pytorch.org/docs/stable/optim.html#module-torch.optim) 的文档


In [ ]:
criterion = nn.NLLLoss()
lr = 0.001
optimizer_vgg = torch.optim.SGD(model_vgg.classifier[6].parameters(),lr = lr)

### 训练模型


In [ ]:
def train_model(model,dataloader,size,epochs=1,optimizer=None):
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        running_corrects = 0
        for inputs,classes in dataloader:
            inputs = inputs.to(device)
            classes = classes.to(device)
            outputs = model(inputs)
            loss = criterion(outputs,classes)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            _,preds = torch.max(outputs.data,1)
            # 统计信息
            running_loss += loss.data.item()
            running_corrects += torch.sum(preds == classes.data)
        epoch_loss = running_loss / size
        epoch_acc = running_corrects.data.item() / size
        print('Loss: {:.4f} Acc: {:.4f}'.format(
                     epoch_loss, epoch_acc))

In [ ]:
%%time
train_model(model_vgg,loader_train,size=dset_sizes['train'],epochs=2,optimizer=optimizer_vgg)

In [ ]:
def test_model(model,dataloader,size):
    model.eval()
    predictions = np.zeros(size)
    all_classes = np.zeros(size)
    all_proba = np.zeros((size,2))
    i = 0
    running_loss = 0.0
    running_corrects = 0
    for inputs,classes in dataloader:
        inputs = inputs.to(device)
        classes = classes.to(device)
        outputs = model(inputs)
        loss = criterion(outputs,classes)           
        _,preds = torch.max(outputs.data,1)
            # 统计信息
        running_loss += loss.data.item()
        running_corrects += torch.sum(preds == classes.data)
        predictions[i:i+len(classes)] = preds.to('cpu').numpy()
        all_classes[i:i+len(classes)] = classes.to('cpu').numpy()
        all_proba[i:i+len(classes),:] = outputs.data.to('cpu').numpy()
        i += len(classes)
    epoch_loss = running_loss / size
    epoch_acc = running_corrects.data.item() / size
    print('Loss: {:.4f} Acc: {:.4f}'.format(
                     epoch_loss, epoch_acc))
    return predictions, all_proba, all_classes

In [ ]:
predictions, all_proba, all_classes = test_model(model_vgg,loader_valid,size=dset_sizes['valid'])

In [ ]:
# 取一个训练数据 batch
inputs, classes = next(iter(loader_valid))

out = torchvision.utils.make_grid(inputs[0:n_images])

imshow(out, title=[dset_classes[x] for x in classes[0:n_images]])

In [ ]:
outputs = model_vgg(inputs[:n_images].to(device))
print(torch.exp(outputs))

In [ ]:
classes[:n_images]

## 通过预计算特征来加速训练

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=3460)

这里你在反复计算同样的量，浪费了大量时间。实际上，VGG 模型的前半部分（叫做 `features`，由卷积层组成）是冻结的，永远不会更新。因此，我们可以为数据集中的每张图片预先算好这些卷积层的输出，因为在训练过程中这些输出始终不变。

下面做的就是这件事。


In [ ]:
x_try = model_vgg.features(inputs_try)

In [ ]:
x_try.shape

可以看到，一张图片计算出的特征形状是 512x7x7（上面是一个 5 张图片的 batch）。


In [ ]:
def preconvfeat(dataloader):
    conv_features = []
    labels_list = []
    for data in dataloader:
        inputs,labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        x = model_vgg.features(inputs)
        conv_features.extend(x.data.cpu().numpy())
        labels_list.extend(labels.data.cpu().numpy())
    conv_features = np.concatenate([[feat] for feat in conv_features])
    return (conv_features,labels_list)

In [ ]:
%%time
conv_feat_train,labels_train = preconvfeat(loader_train)

In [ ]:
conv_feat_train.shape

In [ ]:
%%time
conv_feat_valid,labels_valid = preconvfeat(loader_valid)

### 创建新的数据生成器

我们不再加载图片了，所以需要自己构建数据加载器。如果你看不懂下面这个单元格，没关系！第五课会再讲……


In [ ]:
dtype=torch.float
datasetfeat_train = [[torch.from_numpy(f).type(dtype),torch.tensor(l).type(torch.long)] for (f,l) in zip(conv_feat_train,labels_train)]
datasetfeat_train = [(inputs.reshape(-1), classes) for [inputs,classes] in datasetfeat_train]
loaderfeat_train = torch.utils.data.DataLoader(datasetfeat_train, batch_size=128, shuffle=True)

In [ ]:
%%time
train_model(model_vgg.classifier,dataloader=loaderfeat_train,size=dset_sizes['train'],epochs=50,optimizer=optimizer_vgg)

In [ ]:
datasetfeat_valid = [[torch.from_numpy(f).type(dtype),torch.tensor(l).type(torch.long)] for (f,l) in zip(conv_feat_valid,labels_valid)]
datasetfeat_valid = [(inputs.reshape(-1), classes) for [inputs,classes] in datasetfeat_valid]
loaderfeat_valid = torch.utils.data.DataLoader(datasetfeat_valid, batch_size=128, shuffle=False)

In [ ]:
predictions, all_proba, all_classes = test_model(model_vgg.classifier,dataloader=loaderfeat_valid,size=dset_sizes['valid'])

## 4. 查看模型预测（定性分析）

[视频时间戳](https://youtu.be/ZhC-DIrCe6A?t=3819)

我们最该关注的是验证集上的指标，因为要检查是否过拟合。

用第一个模型，我们应该先设法过拟合，再去操心怎么处理过拟合——如果还在欠拟合，就谈不上正则化、数据增强这些技巧！（这些技术要等两周假期之后再看……）

除了看整体指标，最好也看看每一类的一些例子：

   1. 随机挑几张预测正确的标签
   2. 随机挑几张预测错误的标签
   3. 每个类中置信度最高且预测正确的标签（即概率最高且正确）
   4. 每个类中置信度最高但预测错误的标签（即概率最高但错误）
   5. 最不确定的标签（即概率最接近 0.5 的）

一般来说，这些对排查模型问题特别有用。由于我们的模型非常简单，这个阶段可能学不到太多东西……


In [ ]:
# 每个可视化任务要展示的图片数量
n_view = 8

In [ ]:
correct = np.where(predictions==all_classes)[0]

In [ ]:
len(correct)/dset_sizes['valid']

In [ ]:
from numpy.random import random, permutation
idx = permutation(correct)[:n_view]

In [ ]:
idx

In [ ]:
loader_correct = torch.utils.data.DataLoader([dsets['valid'][x] for x in idx],batch_size = n_view,shuffle=True)

In [ ]:
for data in loader_correct:
    inputs_cor,labels_cor = data

In [ ]:
# 从 batch 生成网格图像
out = torchvision.utils.make_grid(inputs_cor)

imshow(out, title=[l.item() for l in labels_cor])

In [ ]:
from IPython.display import Image, display
for x in idx:
    display(Image(filename=dsets['valid'].imgs[x][0], retina=True))

In [ ]:
incorrect = np.where(predictions!=all_classes)[0]
for x in permutation(incorrect)[:n_view]:
    #print(dsets['valid'].imgs[x][1])
    display(Image(filename=dsets['valid'].imgs[x][0], retina=True))

In [ ]:
##3. 我们最有把握判定为猫、而且确实是猫的图片
correct_cats = np.where((predictions==0) & (predictions==all_classes))[0]
most_correct_cats = np.argsort(all_proba[correct_cats,1])[:n_view]

In [ ]:
for x in most_correct_cats:
    display(Image(filename=dsets['valid'].imgs[correct_cats[x]][0], retina=True))

In [ ]:
##3. 我们最有把握判定为狗、而且确实是狗的图片
correct_dogs = np.where((predictions==1) & (predictions==all_classes))[0]
most_correct_dogs = np.argsort(all_proba[correct_dogs,0])[:n_view]

In [ ]:
for x in most_correct_dogs:
    display(Image(filename=dsets['valid'].imgs[correct_dogs[x]][0], retina=True))

# 结论

我们最后到底做了什么？一个简单的逻辑回归！如果这里联系不起来，下节课我们会用更简单的例子解释。

我们可能是杀鸡用牛刀了！

在这件事里，牛刀就是在大规模 ImageNet 上预训练的 VGG，而 ImageNet 里本来就有大量猫和狗的图片。确实，我们看到未做修改的网络就能预测出猫和狗的品种。因此 VGG 计算出的特征对我们的分类任务非常精准，也就不足为奇了。最后，我们只需要学最后一层线性层的参数，也就是 8194 个参数（别忘了偏置 $2\times 4096+2$）。事实上，这一步在 CPU 上跑也毫无问题。

不过，这个例子仍然很有教学意义，因为它展示了深度学习项目里所有必要的步骤。这里我们没有跟深层网络的训练过程较劲，而是完成了所有前期的工程工作：下载数据集、搭建 GPU 环境、准备数据、用预训练 VGG 计算特征、把特征存到你的硬盘上以备后续实验使用……这些步骤在任何深度学习项目中都是必不可少的，也是你开始愉快地摆弄网络架构、理解训练过程之前必须完成的工作。


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)